In [279]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from scipy import stats
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [234]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [235]:
train_df.set_index("carID")

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.300000,2.0,78.0,0.000000,0.0
6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.000000,1.0,57.0,3.000000,0.0


## Data cleaning

In [236]:
# TODO: Move the functions to a separate file

In [237]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    # Get the most frequent brand for each model -> returns df with model and brand
    brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [238]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [239]:
# Train imputer on train

def fit_imputer(df, fast=True): 
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )

    # TODO: for later
    # Test if we perform better if we use numerical and categorical imputers separately
    
    # Initialize imputer
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # TODO: Test other categorical imputers like: missForest, datawig
    """imputer = IterativeImputer(
        estimator=RandomForestClassifier(),
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="most_frequent"         # Initial fill before iterative process
        )"""

    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships

    imputer.fit(df)
    
    return imputer

In [240]:
def apply_imputer(df, imputer):
    """
        Apply the pretrained imputer to the dataframe
    """

    imputed_values = imputer.transform(df)

    df[df.columns] = imputed_values

    # TODO: rounding is not the best approach as the imputers prediction are continues thus 1.2 doesnt mean the value is closer to 1 than 2
    # However, rounding is the quickest way to fix this for now
    df[["mpg", "engineSize"]] = abs(df[["mpg", "engineSize"]]).round(1)
    try: 
        df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    except KeyError: # will raise keyError if we run it on testing data as it has no price column
        df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    
    return df

In [241]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

### Workflow for Train and Validation Sets

#### Encoding

In [242]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)
# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end

df_encoded, encoders = fit_transform_encoding(df)

#### Creating the stratification column

In [243]:
# Create Categorical price column with 0 < 1 < 2 for the price
df_encoded["price_cat"] = pd.qcut(df_encoded["price"], 3, labels=False)

# Combine the 10 unique brand values (1-9 & NA) with the 3 unique price category values (0-2)
stratify_col = df_encoded["Brand_transformed"].astype(str) + "_" + df_encoded["price_cat"].astype(str)

#### Imputation and decoding

In [244]:
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42, stratify=stratify_col)
# train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputer_test = apply_imputer(validation_split, imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputer_test, encoders)

### Workflow for Seperated Testing Dataset

In [245]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [246]:
test_df.set_index("carID")

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
81363,BMW,X2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
76833,Audi,Q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0


In [247]:
test = simple_processing(test_df)
test_df_encoded, test_encoders = fit_transform_encoding(test)


df_encoded_no_price = df_encoded.drop(["price"], axis=1)
test_imputers = fit_imputer(df_encoded_no_price) # we use the full encoded training dataframe to train the imputers

# Apply trained imputer to both datasplits
test_df_imputed = apply_imputer(test_df_encoded, test_imputers)

test_processed = decode(test_df_imputed, test_encoders)


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- price_cat


## Test of the models on the full dataset

In [248]:
class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "int32" ,"float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")


In [249]:
train_processed = train_processed.set_index("carID")
validation_processed = validation_processed.set_index("carID")
train_processed.index = train_processed.index.astype(int)
validation_processed.index = validation_processed.index.astype(int)


In [250]:
X_train = train_processed.drop(columns=['price'])
y_train = train_processed['price']
X_val = validation_processed.drop(columns=['price'])
y_val = validation_processed['price']

In [251]:
def minimal_features(df):
    df = df.copy()
    df['age'] = 2024 - df['year']
    df = df.drop(columns=["year"])
    df['mileage_per_year'] = df['mileage'] / (df['age'] + 1)
    df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)
    df['age_mileage'] = df['age'] * df['mileage'] / 100000
    df['condition_score'] = (df['paintQuality%']/100) * 0.5 + df['stated_no_damage'].astype(int) * 0.5
    return df

In [252]:
X_train = minimal_features(X_train)
X_val = minimal_features(X_val)

In [257]:
X_train = X_train.drop(columns=["price_cat"])
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 63888 to 27296
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   paintQuality%     60778 non-null  int32  
 5   previousOwners    60778 non-null  int32  
 6   stated_no_damage  60778 non-null  float64
 7   Brand             60778 non-null  object 
 8   transmission      60778 non-null  object 
 9   model             60778 non-null  object 
 10  fuelType          60778 non-null  object 
 11  age               60778 non-null  int32  
 12  mileage_per_year  60778 non-null  float64
 13  efficiency_ratio  60778 non-null  float64
 14  age_mileage       60778 non-null  float64
 15  condition_score   60778 non-null  float64
dtypes: float64(7), int32(5), object(4)
memory

In [258]:
X_val = X_val.drop(columns=["price_cat"])
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15195 entries, 1298 to 71467
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           15195 non-null  int32  
 1   tax               15195 non-null  int32  
 2   mpg               15195 non-null  float64
 3   engineSize        15195 non-null  float64
 4   paintQuality%     15195 non-null  int32  
 5   previousOwners    15195 non-null  int32  
 6   stated_no_damage  15195 non-null  float64
 7   Brand             15195 non-null  object 
 8   transmission      15195 non-null  object 
 9   model             15195 non-null  object 
 10  fuelType          15195 non-null  object 
 11  age               15195 non-null  int32  
 12  mileage_per_year  15195 non-null  float64
 13  efficiency_ratio  15195 non-null  float64
 14  age_mileage       15195 non-null  float64
 15  condition_score   15195 non-null  float64
dtypes: float64(7), int32(5), object(4)
memory 

In [255]:
y_train

carID
63888     7499
9240     22646
39514    26099
35525    31990
24196    11599
         ...  
34859    12990
60621     7999
27292     8999
64574     8998
27296    15950
Name: price, Length: 60778, dtype: int32

In [256]:
y_val

carID
1298     25950
12230     4290
364      33444
11374    18500
43577    25500
         ...  
75377    13499
72130    10510
28075    12750
41700    28980
71467    28995
Name: price, Length: 15195, dtype: int32

In [ ]:
#raise SystemExit("Stop before training the models")

### Linear Regression

In [259]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3365.34
  MAE:  2002.37
  R²:   0.8791

Validation Set Performance (Overall):
  RMSE: 3561.40
  MAE:  2084.90
  R²:   0.8724

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 1540.960148 1072.144568 0.811388
  toyota  3775 1986.246291 1205.567762 0.900533
    ford 13114 2045.523816 1418.846378 0.817998
 hyundai  2724 2095.025698 1474.874706 0.878111
   skoda  3507 2323.678201 1470.741796 0.860267
      vw  8485 2864.281258 1983.197623 0.865078
    audi  5975 4056.966023 2654.139719 0.876011
     bmw  6037 4218.707409 2703.294097 0.866939
mercedes  9527 5556.671846 3377.359562 0.740230

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1594.480139 1111.064006 0.806487
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,1594.480139,1111.064006,0.806487
3,ford,3278,1926.891258,1382.535070,0.835230
8,hyundai,680,2141.823513,1517.375530,0.861959
5,skoda,879,2211.120475,1564.953224,0.878426
7,toyota,943,2238.571676,1287.050070,0.887071
4,vw,2121,2958.811973,2113.360758,0.843025
1,bmw,1509,4124.361375,2826.476447,0.857427
0,audi,1493,4446.625033,2789.536722,0.868045
2,mercedes,2381,6141.186383,3566.336397,0.749027


### ElasticNet

In [260]:
ElasticNet = ElasticNet()
trainer = BrandModelTrainer(ElasticNet)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 4849.31
  MAE:  3001.76
  R²:   0.7489

Validation Set Performance (Overall):
  RMSE: 5083.27
  MAE:  3034.62
  R²:   0.7400

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 2256.262424 1700.984318 0.595643
    ford 13114 2780.895831 2002.023145 0.663614
 hyundai  2724 3405.878632 2451.404966 0.677859
   skoda  3507 3624.201017 2485.958683 0.660084
  toyota  3775 3661.799858 2406.238836 0.661935
      vw  8485 4504.993268 3079.393586 0.666237
    audi  5975 6158.755799 3748.408493 0.714263
mercedes  9527 6647.752365 4275.301978 0.628200
     bmw  6037 7513.730552 4880.810842 0.577913

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 2319.707334 1702.024679 0.590420
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,2319.707334,1702.024679,0.590420
3,ford,3278,2642.565708,1937.377266,0.690104
8,hyundai,680,3268.260449,2374.533849,0.678579
5,skoda,879,3513.438168,2592.535411,0.693042
7,toyota,943,4162.000450,2497.864071,0.609638
4,vw,2121,4252.743259,3048.598875,0.675709
0,audi,1493,7026.041745,3931.945317,0.670553
1,bmw,1509,7027.156854,4683.993637,0.586111
2,mercedes,2381,7625.000108,4558.672907,0.613097


### KNR

In [261]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2354.69
  MAE:  1354.07
  R²:   0.9408

Validation Set Performance (Overall):
  RMSE: 2976.99
  MAE:  1692.24
  R²:   0.9108

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 1089.440096  747.346843 0.905726
    ford 13114 1377.391896  901.485252 0.917475
  toyota  3775 1483.785389  875.260026 0.944492
 hyundai  2724 1548.515768  998.895007 0.933409
   skoda  3507 1861.597180 1126.723524 0.910315
      vw  8485 1999.671460 1335.592905 0.934239
    audi  5975 3040.824603 1941.693456 0.930343
mercedes  9527 3453.464004 2033.805647 0.899661
     bmw  6037 3483.293297 2067.849892 0.909287

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1410.335452  928.513030 0.848603
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,1410.335452,928.513030,0.848603
3,ford,3278,1617.777460,1108.943868,0.883854
8,hyundai,680,1801.499728,1264.967059,0.902342
7,toyota,943,2100.252466,1115.143796,0.900595
5,skoda,879,2156.639720,1479.927418,0.884344
4,vw,2121,2432.382757,1676.847430,0.893913
0,audi,1493,3831.465542,2381.975887,0.902030
1,bmw,1509,4064.438630,2579.761431,0.861539
2,mercedes,2381,4692.831301,2555.926921,0.853448


### Random Forest

In [262]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train, y_train)


rf_trainer.evaluate_train(X_train, y_train)
rf_trainer.evaluate(X_val, y_val)

rf_trainer.evaluate_train_by_brand(X_train, y_train)
rf_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 843.43
  MAE:  475.64
  R²:   0.9924

Validation Set Performance (Overall):
  RMSE: 2181.04
  MAE:  1283.77
  R²:   0.9521

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7634  442.508881 294.069173 0.984446
    ford 13114  526.457985 335.660150 0.987944
 hyundai  2724  551.862154 350.458675 0.991542
  toyota  3775  611.796753 349.407102 0.990563
   skoda  3507  709.810509 421.183852 0.986961
      vw  8485  749.079713 465.700158 0.990772
    audi  5975 1060.519033 655.412408 0.991527
     bmw  6037 1201.302350 673.150823 0.989211
mercedes  9527 1208.062347 690.621601 0.987722

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1202.941227  792.372970 0.889856
    ford 3278 1341.378731

,Brand,N,RMSE,MAE,R²
6,opel,1911,1202.941227,792.372970,0.889856
3,ford,3278,1341.378731,915.902596,0.920151
8,hyundai,680,1352.623800,933.451235,0.944945
7,toyota,943,1449.003415,907.674252,0.952685
5,skoda,879,1736.646487,1153.284516,0.925004
4,vw,2121,1814.461299,1228.350114,0.940967
1,bmw,1509,2863.348558,1777.362797,0.931282
0,audi,1493,3116.717259,1797.472096,0.935173
2,mercedes,2381,3167.258332,1896.249812,0.933244


### Neural Network

In [263]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train, y_train)


mlp_trainer.evaluate_train(X_train, y_train)
mlp_trainer.evaluate(X_val, y_val)


mlp_trainer.evaluate_train_by_brand(X_train, y_train)
mlp_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 60154645.72479516
Validation score: -8.707067
Iteration 2, loss = 59557415.51822785
Validation score: -8.472819
Iteration 3, loss = 56083270.36298810
Validation score: -7.382659
Iteration 4, loss = 44460922.93715557
Validation score: -4.525319
Iteration 5, loss = 23369170.64934541
Validation score: -0.986898
Iteration 6, loss = 8154526.39060133
Validation score: 0.210894
Iteration 7, loss = 4808675.26717637
Validation score: 0.411068
Iteration 8, loss = 3768296.07132576
Validation score: 0.522798
Iteration 9, loss = 3091489.57038666
Validation score: 0.600291
Iteration 10, loss = 2612662.13749583
Validation score: 0.657551
Iteration 11, loss = 2268771.99167037
Validation score: 0.700875
Iteration 12, loss = 1986014.58170769
Validation score: 0.732596
Iteration 13, loss = 1782163.57702544
Validation score: 0.757209
Iteration 14, loss = 1625964.54279743
Validation score: 0.776413
Iteration 15, loss = 1502447.79067565
Validation score: 

,Brand,N,RMSE,MAE,R²
6,opel,1911,1248.186116,827.832603,0.881415
3,ford,3278,1461.299368,1014.117403,0.905236
8,hyundai,680,1525.287906,1043.969092,0.929992
7,toyota,943,1639.527698,997.198514,0.939424
5,skoda,879,1829.390311,1284.397281,0.916780
4,vw,2121,2059.229865,1414.299135,0.923966
1,bmw,1509,2707.257939,1891.161544,0.938569
0,audi,1493,3256.305532,2102.001909,0.929236
2,mercedes,2381,3360.173705,2173.884593,0.924864


| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3561.40 | 2084.90 | 0.8724 |
| ElasticNet | 5083.27 | 3034.62 | 0.7400 |
| KNR|2976.99 |1692.24 |0.9108 |
|RF | 2181.04 |1283.77 |0.9521 |
|NN | 2291.72 |1438.19 |0.9471 |



## Feature selection - Filter Methods

In [264]:
df = X_train.join(y_train)

In [266]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 63888 to 27296
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   paintQuality%     60778 non-null  int32  
 5   previousOwners    60778 non-null  int32  
 6   stated_no_damage  60778 non-null  float64
 7   Brand             60778 non-null  object 
 8   transmission      60778 non-null  object 
 9   model             60778 non-null  object 
 10  fuelType          60778 non-null  object 
 11  age               60778 non-null  int32  
 12  mileage_per_year  60778 non-null  float64
 13  efficiency_ratio  60778 non-null  float64
 14  age_mileage       60778 non-null  float64
 15  condition_score   60778 non-null  float64
 16  price             60778 non-null  int32  

In [267]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
y = df['price']

numerical_cols = df.select_dtypes(include=['int64', "int32",'float64']).columns

# Drop 'carID' and 'price'
num_cols = [col for col in numerical_cols 
            if "_transformed" not in col and col not in ['price', 'carID']]

print(num_cols)
print(categorical_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'condition_score']
['Brand', 'transmission', 'model', 'fuelType']


In [268]:
#FILTER METHOD


#ANOVA FUNCTION

def anova_for_categorical(df, y, categorical_cols):
    # Align indices between df and y
    common_idx = df.index.intersection(y.index)
    df_aligned = df.loc[common_idx]
    y_aligned = y.loc[common_idx]
    
    f_scores, p_values = [], []
    for col in df_aligned.columns:
        if col in categorical_cols:
            groups = [y_aligned[df_aligned[col] == cat] for cat in df_aligned[col].dropna().unique()]
            if len(groups) > 1 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
            else:
                f_stat, p_val = 0.0, 1.0
        else:
            if df_aligned[col].nunique() > 1:
                # Remove NaN values for correlation calculation
                valid_idx = df_aligned[col].notna() & y_aligned.notna()
                if valid_idx.sum() > 1:
                    corr = np.corrcoef(df_aligned.loc[valid_idx, col], y_aligned[valid_idx])[0, 1]
                    f_stat = corr**2 * valid_idx.sum()
                    p_val = 0.0
                else:
                    f_stat, p_val = 0.0, 1.0
            else:
                f_stat, p_val = 0.0, 1.0
        f_scores.append(f_stat)
        p_values.append(p_val)
    
    return np.array(f_scores), np.array(p_values)


def filter_method_selection(X_train, y_train, categorical_cols, num_cols,
                            top_k=None,
                            var_threshold=0.01,
                            corr_threshold=0.85):
    
    print("FILTER METHOD (Variance + Spearman Correlation + ANOVA)")
    
    # Ensure indices match
    common_idx = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_idx]
    y_train = y_train.loc[common_idx]
    
    # Filter numerical columns that exist in X_train
    num_cols_in_X = [col for col in num_cols if col in X_train.columns]
    
    # Variance Threshold
    if num_cols_in_X:
        vt_selector = VarianceThreshold(threshold=var_threshold)
        X_num_vt = pd.DataFrame(
            vt_selector.fit_transform(X_train[num_cols_in_X]),
            columns=np.array(num_cols_in_X)[vt_selector.get_support()],
            index=X_train.index
        )
        print(f"Removed {len(num_cols_in_X) - X_num_vt.shape[1]} low-variance numeric features.")
    else:
        X_num_vt = pd.DataFrame(index=X_train.index)
    
    # Spearman Correlation
    if not X_num_vt.empty and X_num_vt.shape[1] > 1:
        corr_matrix = X_num_vt.corr(method='spearman').abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        X_num_corr = X_num_vt.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} correlated numeric features (Spearman |corr| > {corr_threshold}).")
    else:
        X_num_corr = X_num_vt
    
    # Filter categorical columns that exist in X_train
    categorical_cols_in_X = [col for col in categorical_cols if col in X_train.columns]
    
    # Combine numeric + categorical
    X_filtered = pd.concat([X_num_corr, X_train[categorical_cols_in_X]], axis=1)
    
    # ANOVA F-test
    f_scores, f_pvalues = anova_for_categorical(X_filtered, y_train, categorical_cols_in_X)
    f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)
    
    results_df = pd.DataFrame({
        'feature': X_filtered.columns,
        'ANOVA_F': f_scores,
        'ANOVA_p_value': f_pvalues,
        'ANOVA_norm': f_norm,
        'type': ['categorical' if c in categorical_cols_in_X else 'numerical' for c in X_filtered.columns]
    }).sort_values('ANOVA_norm', ascending=False)
    
    if top_k is None:
        selected_features = X_filtered.columns.tolist()
    else:
        selected_features = X_filtered.columns[np.argsort(f_norm)[-top_k:]].tolist()
    
    print(f"\nFilter method selected {len(selected_features)} features")
    
    return selected_features, X_filtered[selected_features], results_df



In [269]:
#WRAPPER METHOD

def rfe(train_processed, validation_processed, num_cols, step=1, n_estimators=100, random_state=42):
    
    # Filter numeric columns to those that actually exist in train_processed
    valid_num_cols = [col for col in num_cols if col in train_processed.columns]
    if len(valid_num_cols) == 0:
        raise ValueError("No valid numeric columns found in train_processed.")

    print(f"Using {len(valid_num_cols)} numeric columns for RFE:\n{valid_num_cols}")

    # Prepare numeric features and targets
    X_train_num = train_processed[valid_num_cols]
    y_train = train_processed['price']

    X_val_num = validation_processed[valid_num_cols]
    y_val = validation_processed['price']

    nof_list = np.arange(1, X_train_num.shape[1]+1)
    high_score = 0
    nof = 0
    train_score_list = []
    val_score_list = []
    features_to_select = None

    for n in nof_list:
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        rfe = RFE(estimator=model, n_features_to_select=n, step=step)
        X_train_rfe = rfe.fit_transform(X_train_num, y_train)
        X_val_rfe = rfe.transform(X_val_num)
        model.fit(X_train_rfe, y_train)

        train_score = model.score(X_train_rfe, y_train)
        val_score = model.score(X_val_rfe, y_val)
        train_score_list.append(train_score)
        val_score_list.append(val_score)

        if val_score > high_score:
            high_score = val_score
            nof = n
            features_to_select = pd.Series(rfe.support_, index=X_train_num.columns)

    selected_features = features_to_select[features_to_select].index.tolist()
    
    print
    print(f"\nOptimum number of features: {nof}")
    print(f"Best validation score: {high_score:.4f}")
    print("Selected features:")
    print(selected_features)

    return nof, high_score, selected_features, train_score_list, val_score_list


In [270]:
y = df["price"]

df = df.drop(columns=["price"])

In [271]:

#X_train = df[num_cols]
#y_train = df['price']

# Filter method (Variance + Spearman + ANOVA)
# Define categorical and numerical columns first (if not already defined)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['int64',"int32", 'float64']).columns.tolist()

# Filter method with all required parameters
selected_filter, X_train_filtered, filter_df = filter_method_selection(
    df, 
    y,
    categorical_cols=categorical_cols,  # Add this
    num_cols=num_cols,                  # Add this
    top_k=None, 
    var_threshold=0.01, 
    corr_threshold=0.85
)

# Print features selected
print("\nFeatures selected by Filter Method:")
for f in selected_filter:
    print(f)

# Display feature importance scores
print("\nTop 10 Features by ANOVA Score:")
print(filter_df.head(10))

# RFE (Random Forest) using your workflow variables
nof, best_score, selected_rfe_features, train_scores, val_scores = rfe(
    train_processed, validation_processed, num_cols
)


FILTER METHOD (Variance + Spearman Correlation + ANOVA)
Removed 0 low-variance numeric features.
Removed 3 correlated numeric features (Spearman |corr| > 0.85).

Filter method selected 13 features

Features selected by Filter Method:
mileage
tax
mpg
engineSize
paintQuality%
previousOwners
stated_no_damage
age
efficiency_ratio
Brand
transmission
model
fuelType

Top 10 Features by ANOVA Score:
             feature       ANOVA_F  ANOVA_p_value  ANOVA_norm         type
3         engineSize  23022.427577            0.0    1.000000    numerical
7                age  13916.614278            0.0    0.604481    numerical
10      transmission  12785.781433            0.0    0.555362  categorical
0            mileage  10564.372348            0.0    0.458873    numerical
1                tax   6164.934483            0.0    0.267780    numerical
2                mpg   5365.709957            0.0    0.233064    numerical
9              Brand   3134.549183            0.0    0.136152  categorical
12   

In [220]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Test model on reduced DF

In [273]:
X_train = X_train.drop(columns=["mileage_per_year", "age_mileage", "condition_score"])
X_train

,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage,Brand,transmission,model,fuelType,age,efficiency_ratio
carID,,,,,,,,,,,,,
63888,52081,205,43.5,1.6,70,0,1.0,opel,manual,mokka,petrol,9,25.588235
9240,123,145,52.2,1.5,71,2,1.0,bmw,manual,2 series,petrol,5,32.625000
39514,13187,145,51.4,2.2,70,2,1.0,mercedes,manual,cl class,diesel,5,22.347826
35525,7905,145,9.4,2.0,48,1,1.0,mercedes,automatic,e class,diesel,5,4.476190
24196,19064,121,56.2,1.0,70,0,1.0,ford,manual,focus,petrol,6,51.090909
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34859,44000,20,68.9,1.5,94,4,1.0,mercedes,manual,a class,diesel,8,43.062500
60621,43983,160,47.1,1.4,90,3,1.0,opel,manual,astra,petrol,8,31.400000
27292,32496,145,64.2,1.1,88,4,1.0,ford,manual,fiesta,petrol,6,53.500000


In [274]:
X_val = X_val.drop(columns=["mileage_per_year", "age_mileage", "condition_score"])
X_val

,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage,Brand,transmission,model,fuelType,age,efficiency_ratio
carID,,,,,,,,,,,,,
1298,2888,150,44.8,1.5,67,1,1.0,audi,semi-auto,a1,petrol,4,28.000000
12230,97000,316,29.7,2.5,73,4,1.0,bmw,manual,5 series,petrol,18,11.423077
364,475,145,31.4,2.0,58,4,1.0,audi,semi-auto,q3,petrol,5,14.952381
11374,20148,150,60.1,2.0,58,0,1.0,bmw,automatic,1 series,diesel,7,28.619048
43577,13940,145,56.5,2.1,86,4,1.0,mercedes,automatic,glc class,diesel,6,25.681818
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75377,46421,20,67.3,2.0,94,0,1.0,vw,manual,golf,diesel,10,32.047619
72130,17137,145,60.1,1.2,99,2,1.0,vw,manual,polo,petrol,7,46.230769
28075,3445,145,56.5,1.0,50,2,1.0,ford,manual,fiesta,petrol,6,51.363636


In [275]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 3482.42
  MAE:  2105.05
  R²:   0.8705

Validation Set Performance (Overall):
  RMSE: 3706.33
  MAE:  2192.68
  R²:   0.8618

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 1576.610692 1099.136635 0.802560
  toyota  3775 2000.701226 1207.959417 0.899080
    ford 13114 2129.309495 1515.909266 0.802782
 hyundai  2724 2185.692500 1525.782041 0.867332
   skoda  3507 2387.376164 1515.046435 0.852501
      vw  8485 2934.201728 2031.412452 0.858411
    audi  5975 4200.802551 2761.738650 0.867063
     bmw  6037 4472.901426 2979.855711 0.850421
mercedes  9527 5715.809066 3559.747573 0.725138

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1635.606951 1130.223396 0.796376
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,1635.606951,1130.223396,0.796376
3,ford,3278,2034.696832,1487.588322,0.816277
7,toyota,943,2238.849995,1285.012811,0.887043
8,hyundai,680,2253.635423,1573.959034,0.847170
5,skoda,879,2271.814399,1609.709619,0.871661
4,vw,2121,2957.936075,2135.165391,0.843118
1,bmw,1509,4426.138031,3110.754008,0.835799
0,audi,1493,4657.624158,2912.633829,0.855225
2,mercedes,2381,6376.904165,3785.510456,0.729391


In [280]:
Elastic = ElasticNet()
trainer = BrandModelTrainer(Elastic)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 4912.36
  MAE:  3030.20
  R²:   0.7424

Validation Set Performance (Overall):
  RMSE: 5145.46
  MAE:  3064.56
  R²:   0.7336

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 2290.272625 1725.510675 0.583361
    ford 13114 2798.942471 2016.065877 0.659234
 hyundai  2724 3453.459172 2464.973434 0.668796
   skoda  3507 3635.749005 2485.325297 0.657914
  toyota  3775 3679.830424 2395.760843 0.658598
      vw  8485 4559.447473 3118.650894 0.658119
    audi  5975 6281.723911 3801.759992 0.702738
mercedes  9527 6720.458644 4311.353165 0.620023
     bmw  6037 7630.041884 4941.549209 0.564744

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 2356.468710 1726.174840 0.577336
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,2356.468710,1726.174840,0.577336
3,ford,3278,2665.467683,1959.247391,0.684709
8,hyundai,680,3346.541560,2417.094983,0.662997
5,skoda,879,3535.496469,2609.382820,0.689176
7,toyota,943,4169.982633,2498.388190,0.608139
4,vw,2121,4304.691010,3094.580435,0.667738
0,audi,1493,7080.355314,3966.736783,0.665440
1,bmw,1509,7137.374919,4693.822894,0.573026
2,mercedes,2381,7732.848925,4612.628968,0.602074


In [281]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 2356.35
  MAE:  1347.28
  R²:   0.9407

Validation Set Performance (Overall):
  RMSE: 2982.84
  MAE:  1698.25
  R²:   0.9105

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7634 1086.593424  744.584700 0.906218
    ford 13114 1346.642861  875.125667 0.921119
  toyota  3775 1379.041889  840.160954 0.952053
 hyundai  2724 1548.687673  996.852056 0.933394
   skoda  3507 1834.625003 1104.289022 0.912895
      vw  8485 1937.444296 1315.468097 0.938268
    audi  5975 3091.607501 1970.468954 0.927997
     bmw  6037 3464.570799 2069.766076 0.910259
mercedes  9527 3515.310822 2050.423953 0.896035

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1445.001918  942.589639 0.841069
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1911,1445.001918,942.589639,0.841069
3,ford,3278,1566.111475,1082.354912,0.891155
7,toyota,943,1796.447785,1042.239024,0.927274
8,hyundai,680,1918.201907,1338.416471,0.889279
5,skoda,879,2054.382785,1407.954949,0.895051
4,vw,2121,2374.194272,1660.361433,0.898928
0,audi,1493,3781.435077,2397.951373,0.904572
1,bmw,1509,4053.797032,2579.177999,0.862264
2,mercedes,2381,4841.328655,2659.142545,0.844026


In [282]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train, y_train)


rf_trainer.evaluate_train(X_train, y_train)
rf_trainer.evaluate(X_val, y_val)

rf_trainer.evaluate_train_by_brand(X_train, y_train)
rf_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ opel done.
  ✓ bmw done.
  ✓ mercedes done.
  ✓ ford done.
  ✓ hyundai done.
  ✓ audi done.
  ✓ vw done.
  ✓ toyota done.
  ✓ skoda done.

Training Set Performance (Overall):
  RMSE: 838.94
  MAE:  473.76
  R²:   0.9925

Validation Set Performance (Overall):
  RMSE: 2187.58
  MAE:  1275.76
  R²:   0.9518

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7634  443.518523 295.062130 0.984375
    ford 13114  519.874693 334.934903 0.988244
 hyundai  2724  553.520123 353.923524 0.991491
  toyota  3775  609.711715 351.698260 0.990627
   skoda  3507  718.481529 421.989695 0.986641
      vw  8485  738.499252 463.694797 0.991031
    audi  5975 1061.917583 653.070576 0.991505
     bmw  6037 1192.293579 661.630018 0.989372
mercedes  9527 1200.688419 687.183910 0.987871

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1911 1191.189926  782.571315 0.891997
    ford 3278 1348.012084

,Brand,N,RMSE,MAE,R²
6,opel,1911,1191.189926,782.571315,0.891997
3,ford,3278,1348.012084,916.765290,0.919360
7,toyota,943,1427.185559,894.440615,0.954099
8,hyundai,680,1427.947505,953.384426,0.938643
5,skoda,879,1734.366781,1153.201547,0.925201
4,vw,2121,1797.426286,1223.217151,0.942071
1,bmw,1509,2869.363358,1754.478622,0.930993
0,audi,1493,3097.445932,1771.275111,0.935972
2,mercedes,2381,3207.474059,1886.880769,0.931538


In [283]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train, y_train)


mlp_trainer.evaluate_train(X_train, y_train)
mlp_trainer.evaluate(X_val, y_val)


mlp_trainer.evaluate_train_by_brand(X_train, y_train)
mlp_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 59987479.44637313
Validation score: -8.819786
Iteration 2, loss = 59129313.35862910
Validation score: -8.472948
Iteration 3, loss = 54315728.02271126
Validation score: -6.940425
Iteration 4, loss = 39195965.28358083
Validation score: -3.380844
Iteration 5, loss = 16672480.13930610
Validation score: -0.388361
Iteration 6, loss = 6649716.26585727
Validation score: 0.210376
Iteration 7, loss = 4345250.76080136
Validation score: 0.434608
Iteration 8, loss = 3334480.72374015
Validation score: 0.560741
Iteration 9, loss = 2728277.66600703
Validation score: 0.639235
Iteration 10, loss = 2317499.82149711
Validation score: 0.691055
Iteration 11, loss = 2037579.57426730
Validation score: 0.727681
Iteration 12, loss = 1822235.09284322
Validation score: 0.752926
Iteration 13, loss = 1662089.02666896
Validation score: 0.771688
Iteration 14, loss = 1535469.90927962
Validation score: 0.785455
Iteration 15, loss = 1446002.27492717
Validation score: 

,Brand,N,RMSE,MAE,R²
6,opel,1911,1266.101404,843.185541,0.877986
3,ford,3278,1478.690136,1020.846857,0.902967
8,hyundai,680,1601.605005,1079.380375,0.922812
7,toyota,943,1730.988363,1019.557526,0.932477
5,skoda,879,1795.319735,1277.087609,0.919851
4,vw,2121,2012.093250,1389.888719,0.927407
1,bmw,1509,2720.322555,1877.435189,0.937975
0,audi,1493,2984.054912,1956.621582,0.940574
2,mercedes,2381,3572.847051,2254.475498,0.915052


Performance on the full dataset

| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3561.40 | 2084.90 | 0.8724 |
| ElasticNet        | 5083.27 | 3034.62 | 0.7400 |
| KNR               | 2976.99 | 1692.24 | 0.9108 |
| RF                | **2181.04** | **1283.77** | **0.9521** |
| NN                | 2291.72 | 1438.19 | 0.9471 |

Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3706.33 | 2192.68 | 0.8618 |
| ElasticNet        | 5145.46 | 3064.56 | 0.7336 |
| KNR               | 2982.84 | 1698.25 | 0.9105 |
| RF                | **2187.58** | **1275.76** | **0.9518** |
| NN                | 2309.92 | 1437.70 | 0.9436 |




## Feature Selection - Wrapper Method

## PCA

## Test models on PCA

## Hyperparam tuning

## predictions